# 06b — Governança e Segurança com Unity Catalog
## Módulo 6 | Seção do Exame: S7 (15%)

**O que este notebook faz:**
- `GRANT`, `REVOKE` e hierarquia de permissões do UC
- Diferença prática entre **Managed Tables** e **External Tables**
- **Row-Level Security** via Row Filter
- **Column Masking** via Masking Policy

> **Nota:** Algumas operações exigem um segundo usuário para demonstração completa.  
> Se você está sozinho no Free Edition, execute os GRANTs normalmente — o entendimento do  
> mecanismo é suficiente para o exame.

---

### ✅ Checklist deste notebook

**GRANT / REVOKE:**
- [ ] `GRANT USE CATALOG` executado
- [ ] `GRANT SELECT ON TABLE` executado
- [ ] `REVOKE SELECT` executado
- [ ] `SHOW GRANTS` confirmou estado das permissões
- [ ] Entendi: preciso conceder nos 3 níveis (catálogo + schema + tabela)

**Managed vs External Tables:**
- [ ] Managed table criada sem LOCATION
- [ ] External table: sem external location, criada como managed (ver célula 14)
- [ ] `DESCRIBE EXTENDED` mostrou diferença no campo Location
- [ ] `DROP TABLE` managed → dados físicos apagados
- [ ] External table (com external location): DROP preserva dados — não demonstrado (sem S3)

**Row Filter + Column Mask:**
- [ ] Row filter function criada
- [ ] `ALTER TABLE SET ROW FILTER` aplicado
- [ ] Column mask function criada
- [ ] `ALTER TABLE ALTER COLUMN SET MASK` aplicado

In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")
CATALOG      = dbutils.widgets.get("catalog")
VOLUME_PATH  = f"/Volumes/{CATALOG}/bronze/landing"

# Seu e-mail de usuário no Databricks Free Edition
# Substitua pelo e-mail real para testar GRANT em si mesmo
import subprocess
try:
    CURRENT_USER = spark.sql("SELECT current_user()").collect()[0][0]
    print(f"Usuário atual: {CURRENT_USER}")
except Exception:
    CURRENT_USER = "anastudies.data@gmail.com"
    print(f"Substitua CURRENT_USER pelo seu e-mail: {CURRENT_USER}")

---
# SEÇÃO 1 — GRANT, REVOKE e Hierarquia do UC

**Hierarquia de permissões do Unity Catalog:**
```
Metastore
  └── Catalog  ← GRANT USE CATALOG
        └── Schema  ← GRANT USE SCHEMA
              └── Table  ← GRANT SELECT / MODIFY / ALL PRIVILEGES
```

**Regra crítica do exame:** Para um usuário ler uma tabela, ele precisa de permissão em **todos os 3 níveis**.  
`GRANT SELECT ON TABLE` sem `USE CATALOG` + `USE SCHEMA` → acesso negado.

In [0]:
# Nível 1: acesso ao catálogo
# Substitua <user@email.com> pelo e-mail de um segundo usuário ou o seu próprio
spark.sql(f"GRANT USE CATALOG ON CATALOG workshop_weather TO `{CURRENT_USER}`")

In [0]:
# Nível 2: acesso ao schema Gold
spark.sql(f"GRANT USE SCHEMA ON SCHEMA workshop_weather.gold TO `{CURRENT_USER}`")

In [0]:
# Nível 3: acesso somente leitura à tabela daily_summary
spark.sql(f"GRANT SELECT ON TABLE workshop_weather.gold.daily_summary TO `{CURRENT_USER}`")

In [0]:
%sql
-- Verifica o estado atual das permissões na tabela
SHOW GRANTS ON TABLE workshop_weather.gold.daily_summary;

In [0]:
# Remove o acesso SELECT
spark.sql(f"REVOKE SELECT ON TABLE workshop_weather.gold.daily_summary FROM `{CURRENT_USER}`")

In [0]:
%sql
-- Confirma que a permissão foi removida
SHOW GRANTS ON TABLE workshop_weather.gold.daily_summary;

---
# SEÇÃO 2 — Managed Tables vs External Tables

**Diferença fundamental:**

| | Managed Table | External Table |
|---|---|---|
| Criação | `CREATE TABLE ... USING DELTA` (sem LOCATION) | `CREATE TABLE ... LOCATION '/caminho'` |
| Storage | Gerenciado pelo UC (path automático) | Path externo (Volume, ADLS, S3) |
| `DROP TABLE` | Apaga metadados **E** dados físicos | Apaga apenas metadados — dados permanecem |
| Uso típico | Tabelas criadas e gerenciadas pelo UC | Dados compartilhados entre sistemas |

In [0]:
%sql
-- MANAGED TABLE: UC controla o ciclo de vida dos dados
CREATE TABLE IF NOT EXISTS workshop_weather.gold.managed_test (
    id    INT,
    valor STRING
)
USING DELTA
COMMENT 'Tabela gerenciada — dados apagados junto com os metadados no DROP';

INSERT INTO workshop_weather.gold.managed_test VALUES (1, 'teste managed');

In [0]:
%sql
-- Inspeciona o Location da managed table
-- O path será algo como: dbfs:/user/hive/warehouse/workshop_weather.db/gold.db/managed_test
DESCRIBE EXTENDED workshop_weather.gold.managed_test;

In [0]:
# EXTERNAL TABLE: dados em um path explícito que você controla
external_path = f"{VOLUME_PATH}/external_test"
print(f"Path da external table: {external_path}")

In [0]:
# UC Volumes são managed storage — não suportam external tables com LOCATION.
# Para criar uma external table, seria necessário um external location registrado no UC.
# Criamos a tabela aqui para continuar o workshop (células 15 e 16 dependem dela).
df = spark.createDataFrame([(1, "teste external")], "id INT, valor STRING")
df.write.format("delta").mode("overwrite").saveAsTable("workshop_weather.gold.external_test")
print("✅ Tabela criada e populada.")
print("   ℹ️  External tables no UC exigem um external location, não um Volume.")

In [0]:
%sql
-- Inspeciona o Location da tabela external_test
-- ℹ️  Sem external location disponível, esta tabela foi criada como MANAGED no UC.
-- O path é gerenciado pelo UC (não é o Volume definido acima).
DESCRIBE EXTENDED workshop_weather.gold.external_test;

In [0]:
# Demonstração do comportamento do DROP TABLE

# DROP da managed table — dados físicos serão apagados
spark.sql("DROP TABLE IF EXISTS workshop_weather.gold.managed_test")
print("✅ managed_test dropada.")
print("   → Os dados físicos foram apagados junto com os metadados.")
print()

# DROP da external_test — também é managed neste workshop (sem external location)
spark.sql("DROP TABLE IF EXISTS workshop_weather.gold.external_test")
print("✅ external_test dropada.")
print("   → Como não há external location, esta tabela também é managed.")
print("   → Os dados físicos foram apagados junto com os metadados.")
print()
print("ℹ️  Para demonstrar external table real (dados preservados após DROP),")
print("   seria necessário registrar um external location no UC com um bucket S3 próprio.")

---
# SEÇÃO 3 — Row-Level Security e Column Masking

**Row Filter:** Restringe quais **linhas** cada usuário pode ver.  
**Column Mask:** Restringe quais **valores** de uma coluna são visíveis.

Ambos são centralizados no UC — aplicam-se a **todos os acessos** automaticamente:  
SQL Editor, notebooks, dashboards, BI tools e ML.

In [0]:
%sql
-- Row Filter: cada usuário vê apenas os dados da sua cidade
-- A função retorna TRUE (linha visível) se:
--   - o usuário é admin (is_member('admins')), OU
--   - a city_name contém o nome do usuário atual (simulação didática)
-- Em produção real, você mapearia usuário → cidade em uma tabela de controle.

CREATE OR REPLACE FUNCTION workshop_weather.silver.filter_by_city(city_name STRING)
RETURN
    is_member('admins')  -- admins veem tudo
    OR city_name = 'Sao Paulo'  -- substituir por lógica real de mapeamento
;

In [0]:
%sql
-- Aplica o row filter na tabela Silver
-- A partir daqui, cada query filtrará linhas automaticamente baseado na função
ALTER TABLE workshop_weather.silver.current_weather
SET ROW FILTER workshop_weather.silver.filter_by_city ON (city_name);

In [0]:
%sql
-- Testa o row filter: um admin vê todas as cidades
-- Um usuário não-admin veria apenas os dados filtrados pela função
SELECT city_name, temp, weather_main
FROM workshop_weather.silver.current_weather
ORDER BY city_name;

In [0]:
%sql
-- Remove o row filter para não impactar os outros notebooks
ALTER TABLE workshop_weather.silver.current_weather
DROP ROW FILTER;

In [0]:
%sql
-- Column Mask: oculta a umidade (humidity) para não-admins
-- Admins veem o valor real; outros usuários veem NULL

CREATE OR REPLACE FUNCTION workshop_weather.silver.mask_humidity(humidity INT)
RETURN
    CASE
        WHEN is_member('admins') THEN humidity  -- admin: valor real
        ELSE NULL                               -- outros: NULL (dado mascarado)
    END
;

In [0]:
%sql
-- Aplica o column mask na coluna humidity
ALTER TABLE workshop_weather.silver.current_weather
ALTER COLUMN humidity SET MASK workshop_weather.silver.mask_humidity;

In [0]:
%sql
-- Como admin, vê o valor real de humidity
-- Um usuário não-admin veria NULL no lugar
SELECT city_name, temp, humidity, weather_main
FROM workshop_weather.silver.current_weather
ORDER BY city_name;

In [0]:
%sql
-- Remove o column mask para não impactar os outros notebooks
ALTER TABLE workshop_weather.silver.current_weather
ALTER COLUMN humidity DROP MASK;

---
## ✅ Checklist Final — Módulo 6 concluído?

**GRANT / REVOKE:**
- [ ] Executei GRANT nos 3 níveis: catálogo + schema + tabela
- [ ] REVOKE removeu o acesso
- [ ] Entendi: sem USE CATALOG, SELECT na tabela não funciona

**Managed vs External:**
- [ ] Managed: DROP apaga dados físicos
- [ ] External: sem external location disponível — não demonstrado (requer S3)
- [ ] `DESCRIBE EXTENDED` mostra Location gerenciado pelo UC (ambas managed)

**Row Filter + Column Mask:**
- [ ] Row filter aplicado e removido — entendi o mecanismo
- [ ] Column mask aplicado e removido — entendi o mecanismo
- [ ] Entendi que ambos são aplicados automaticamente em todos os acessos (BI, ML, SQL)

**Próximo: configurar o Workflows Job na UI do Databricks (Sub-tarefa 5.1 do plano)**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*